# Step 9 — Baseline Logistic Regression

A logistic regression model was developed as an interpretable baseline
for predicting clinical deterioration occurring between 2 and 8 hours
after ICU admission using information available during the first 2 hours.

To reduce information leakage, data splitting was performed at the
patient level using `subject_id`, ensuring that ICU stays belonging to
the same patient could not occur in both training and test sets.

Missing-value imputation and feature scaling were performed within a
scikit-learn pipeline so that preprocessing parameters were estimated
from the training data only.

Because the outcome was imbalanced, class weighting was used rather than
synthetic oversampling.

Given the small MIMIC-IV Demo sample, all performance estimates are
exploratory and should not be interpreted as evidence of clinical
validity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)

DATA_PATH = "../results/model_features_2h.csv"

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("STEP 9 — BASELINE LOGISTIC REGRESSION")
print("=" * 70)

print("Dataset shape:", df.shape)
print("ICU stays:", df["stay_id"].nunique())
print("Patients:", df["subject_id"].nunique())

print("\nOutcome:")
print(df["future_deterioration"].value_counts())

ID_COLUMNS = [
    "subject_id",
    "hadm_id",
    "stay_id"
]

TARGET = "future_deterioration"

feature_columns = [
    col for col in df.columns
    if col not in ID_COLUMNS + [TARGET]
]

X = df[feature_columns].copy()
y = df[TARGET].copy()
groups = df["subject_id"].copy()

print("Predictors:", X.shape[1])
print("Outcome observations:", len(y))
print("Unique patient groups:", groups.nunique())

all_missing_columns = [
    col for col in X.columns
    if X[col].isna().all()
]

print(
    "Completely missing predictors:",
    all_missing_columns
)

if len(all_missing_columns) > 0:

    X = X.drop(
        columns=all_missing_columns
    )

feature_columns = X.columns.tolist()

print(
    "Predictors after empty-column check:",
    X.shape[1]
)

all_missing_columns = [
    col for col in X.columns
    if X[col].isna().all()
]

print(
    "Completely missing predictors:",
    all_missing_columns
)

if len(all_missing_columns) > 0:

    X = X.drop(
        columns=all_missing_columns
    )

feature_columns = X.columns.tolist()

print(
    "Predictors after empty-column check:",
    X.shape[1]
)

# Create a patient-level train/test split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_groups = groups.iloc[train_idx].copy()
test_groups = groups.iloc[test_idx].copy()

patient_overlap = set(train_groups.unique()) & set(test_groups.unique())

assert not patient_overlap, (
    f"Patient leakage detected: {patient_overlap}"
)

print("Split completed successfully.")

print("=" * 70)
print("PATIENT-LEVEL SPLIT")
print("=" * 70)

print("Training ICU stays:", len(X_train))
print("Test ICU stays:", len(X_test))

print(
    "Training patients:",
    train_groups.nunique()
)

print(
    "Test patients:",
    test_groups.nunique()
)

print(
    "Patient overlap:",
    len(patient_overlap)
)

print("\nTraining outcome:")
print(y_train.value_counts())

print("\nTest outcome:")
print(y_test.value_counts())

logistic_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=5000,
                random_state=42
            )
        )
    ]
)

logistic_pipeline

logistic_pipeline.fit(
    X_train,
    y_train
)

print(
    "Baseline logistic regression fitted successfully."
)

y_pred = logistic_pipeline.predict(
    X_test
)

y_prob = logistic_pipeline.predict_proba(
    X_test
)[:, 1]

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm)

print("\nTrue negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

plt.figure(figsize=(5, 4))

plt.imshow(cm)

plt.xticks(
    [0, 1],
    ["No deterioration", "Deterioration"]
)

plt.yticks(
    [0, 1],
    ["No deterioration", "Deterioration"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression Confusion Matrix")

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

accuracy = accuracy_score(
    y_test,
    y_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else np.nan
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

print("=" * 70)
print("CLASSIFICATION METRICS")
print("=" * 70)

print(
    "Accuracy:",
    round(accuracy, 3)
)

print(
    "Balanced accuracy:",
    round(balanced_accuracy, 3)
)

print(
    "Precision:",
    round(precision, 3)
)

print(
    "Sensitivity / Recall:",
    round(recall, 3)
)

print(
    "Specificity:",
    round(specificity, 3)
)

print(
    "F1 score:",
    round(f1, 3)
)

if y_test.nunique() == 2:

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    pr_auc = average_precision_score(
        y_test,
        y_prob
    )

    brier = brier_score_loss(
        y_test,
        y_prob
    )

    print(
        "AUROC:",
        round(roc_auc, 3)
    )

    print(
        "Average precision / PR-AUC:",
        round(pr_auc, 3)
    )

    print(
        "Brier score:",
        round(brier, 3)
    )

else:

    roc_auc = np.nan
    pr_auc = np.nan
    brier = np.nan

    print(
        "AUROC/PR-AUC cannot be reliably "
        "calculated because the test set "
        "contains only one outcome class."
    )

    if y_test.nunique() == 2:

        fpr, tpr, thresholds = roc_curve(
        y_test,
        y_prob
    )

    plt.figure(figsize=(6, 5))

    plt.plot(
        fpr,
        tpr,
        label=f"Logistic regression (AUROC={roc_auc:.2f})"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )

    plt.xlabel(
        "False Positive Rate"
    )

    plt.ylabel(
        "True Positive Rate"
    )

    plt.title(
        "ROC Curve — Baseline Logistic Regression"
    )

    plt.legend()

    plt.tight_layout()
    plt.show()

    if y_test.nunique() == 2:

        precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y_test,
            y_prob
        )
    )

    baseline_prevalence = (
        y_test.mean()
    )

    plt.figure(figsize=(6, 5))

    plt.plot(
        recall_curve,
        precision_curve,
        label=f"Logistic regression (AP={pr_auc:.2f})"
    )

    plt.axhline(
        baseline_prevalence,
        linestyle="--",
        label=f"Test prevalence={baseline_prevalence:.2f}"
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")

    plt.title(
        "Precision–Recall Curve"
    )

    plt.legend()

    plt.tight_layout()
    plt.show()

    print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "No deterioration",
            "Deterioration"
        ],
        zero_division=0
    )
)

model = (
    logistic_pipeline
    .named_steps["model"]
)

coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_[0]
})

coefficients[
    "absolute_coefficient"
] = (
    coefficients[
        "coefficient"
    ].abs()
)

coefficients = (
    coefficients
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
)

display(
    coefficients.head(20)
)

coefficients.to_csv(
    "../results/logistic_coefficients.csv",
    index=False
)

test_predictions = df.iloc[
    test_idx
][
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        TARGET
    ]
].copy()

test_predictions[
    "predicted_probability"
] = y_prob

test_predictions[
    "predicted_class"
] = y_pred

test_predictions[
    "prediction_correct"
] = (
    test_predictions[TARGET]
    == test_predictions["predicted_class"]
)

display(
    test_predictions
    .sort_values(
        "predicted_probability",
        ascending=False
    )
)

test_predictions.to_csv(
    "../results/logistic_test_predictions.csv",
    index=False
)

logistic_results = pd.DataFrame({
    "metric": [
        "Training ICU stays",
        "Test ICU stays",
        "Training patients",
        "Test patients",
        "Patient overlap",
        "Test positives",
        "Test negatives",
        "Accuracy",
        "Balanced accuracy",
        "Precision",
        "Sensitivity",
        "Specificity",
        "F1",
        "AUROC",
        "PR-AUC",
        "Brier score"
    ],

    "value": [
        len(X_train),
        len(X_test),
        train_groups.nunique(),
        test_groups.nunique(),
        len(patient_overlap),
        int(y_test.sum()),
        int((y_test == 0).sum()),
        accuracy,
        balanced_accuracy,
        precision,
        recall,
        specificity,
        f1,
        roc_auc,
        pr_auc,
        brier
    ]
})

display(logistic_results)

print("\n")
print("=" * 70)
print("STEP 9 — BASELINE LOGISTIC REGRESSION COMPLETE")
print("=" * 70)

print(
    "Total ICU stays:",
    len(df)
)

print(
    "Total patients:",
    df["subject_id"].nunique()
)

print(
    "Training ICU stays:",
    len(X_train)
)

print(
    "Test ICU stays:",
    len(X_test)
)

print(
    "Training patients:",
    train_groups.nunique()
)

print(
    "Test patients:",
    test_groups.nunique()
)

print(
    "Patient overlap:",
    len(patient_overlap)
)

print(
    "Test positive cases:",
    int(y_test.sum())
)

print(
    "Test negative cases:",
    int((y_test == 0).sum())
)

print(
    "Accuracy:",
    round(accuracy, 3)
)

print(
    "Balanced accuracy:",
    round(balanced_accuracy, 3)
)

print(
    "Sensitivity:",
    round(recall, 3)
)

print(
    "Specificity:",
    round(specificity, 3)
)

print(
    "AUROC:",
    round(roc_auc, 3)
    if not np.isnan(roc_auc)
    else "NA"
)

print(
    "PR-AUC:",
    round(pr_auc, 3)
    if not np.isnan(pr_auc)
    else "NA"
)

print(
    "Brier score:",
    round(brier, 3)
    if not np.isnan(brier)
    else "NA"
)

print("\nNo patient overlap allowed.")
print("Imputation learned from training data only.")
print("Scaling learned from training data only.")
print("No SMOTE used.")